# Tugas 4: Implementasi Word Embedding Skip-gram dengan Artificial Neural Network (ANN) & Gensim

Pada tugas ini, kita mempelajari dan mengimplementasikan model **Word2Vec arsitektur Skip-gram** menggunakan:
1. **Artificial Neural Network (ANN)** yang dibangun dari awal (*from scratch*) dengan Python & NumPy murni untuk memahami mekanisme komputasi internal (Forward-Backward Propagation dan Gradient Descent).
2. **Library Gensim** yang memungkinkan pengaturan parameter dimensi vektor (`vector_size`), ukuran jendela (*window size*), dan jumlah epoch secara fleksibel.

---

### Alur Kerja :
1. **Penyusunan Kumpulan Kalimat & Ekstraksi Kumpulan Kata (Vocabulary)**.
2. **Representasi One-Hot Encoding** untuk setiap kata unik.
3. **Pembentukan Pasangan Data Latih (Target & Context Pairs)** menggunakan *Sliding Window* ($C = 3$, bisa disesuaikan).
4. **Desain Arsitektur ANN Skip-gram** (Input Layer, Projection/Hidden Layer, Output Layer Softmax).
5. **Mekanisme Forward & Backward Propagation** serta penurunan rumus turunan gradien (SGD).
6. **Pelatihan Model ANN** dan visualisasi kurva fungsi rugi (*loss curve*).
7. **Ekstraksi Word Vectors (Embedding)** dan analisis kemiripan semantik (*Cosine Similarity*).
8. **Representasi 3 Kalimat Menjadi Vektor (Sentence Vector Representation)** melalui teknik *Mean Pooling*.
9. **Visualisasi Ruang Vektor 2D** menggunakan PCA.
10. **Implementasi Skip-gram dengan Library Gensim** (Parameter Dimensi Bebas Diatur).

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed agar hasil eksperimen konsisten (reproducible)
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Pustaka NumPy, Pandas, Matplotlib, dan Seaborn berhasil dimuat!")

Pustaka NumPy, Pandas, Matplotlib, dan Seaborn berhasil dimuat!


## 1. Pembuatan Kumpulan Kalimat & Ekstraksi Kumpulan Kata (Vocabulary)

Kita menyiapkan 3 kalimat yang memuat topik informatika, olahraga, dan teknologi komputer:
1. **Kalimat 1**: *"mahasiswa informatika belajar kecerdasan buatan"*
2. **Kalimat 2**: *"pemain sepak bola mencetak gol kemenangan tim"*
3. **Kalimat 3**: *"program komputer menggunakan algoritma cerdas"*

In [2]:
# 1. Definisi 3 Kalimat
kalimat_list = [
    "mahasiswa informatika belajar kecerdasan buatan",
    "pemain sepak bola mencetak gol kemenangan tim",
    "program komputer menggunakan algoritma cerdas"
]

print("=== DAFTAR KALIMAT ===")
for i, kal in enumerate(kalimat_list, 1):
    print(f"Kalimat {i}: '{kal}'")

# 2. Tokenisasi teks menjadi kumpulan token kata per kalimat
tokenized_sentences = [kal.lower().split() for kal in kalimat_list]

# 3. Ekstraksi Vocabulary (Kumpulan Seluruh Kata Unik)
vocab = sorted(list(set(word for sentence in tokenized_sentences for word in sentence)))
vocab_size = len(vocab)

# 4. Peta Indeks Kata (Word-to-Index dan Index-to-Word)
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for idx, word in enumerate(vocab)}

print(f"\nJumlah kata unik (Vocabulary Size / V): {vocab_size} kata\n")
df_vocab = pd.DataFrame({
    'Indeks': list(word2idx.values()),
    'Kata (Token)': list(word2idx.keys())
})
display(df_vocab)

=== DAFTAR KALIMAT ===
Kalimat 1: 'mahasiswa informatika belajar kecerdasan buatan'
Kalimat 2: 'pemain sepak bola mencetak gol kemenangan tim'
Kalimat 3: 'program komputer menggunakan algoritma cerdas'

Jumlah kata unik (Vocabulary Size / V): 17 kata



,Indeks,Kata (Token)
0,0,algoritma
1,1,belajar
2,2,bola
3,3,buatan
4,4,cerdas
5,5,gol
6,6,informatika
7,7,kecerdasan
8,8,kemenangan
9,9,komputer


## 2. Representasi One-Hot Encoding

Setiap kata dari vocabulary berukuran $V=17$ direpresentasikan sebagai sebuah vektor biner $x \in \{0, 1\}^V$, di mana hanya elemen pada posisi indeks kata tersebut yang bernilai 1:

$$x_i = \begin{cases} 1, & \text{jika } i = \text{indeks kata} \\ 0, & \text{lainnya} \end{cases}$$

In [3]:
def get_one_hot(word, word2idx, vocab_size):
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1.0
    return vec

# Buat matriks one-hot untuk seluruh kata dalam vocabulary
one_hot_matrix = np.array([get_one_hot(w, word2idx, vocab_size) for w in vocab])
df_one_hot = pd.DataFrame(one_hot_matrix, index=vocab, columns=[f"dim_{i}" for i in range(vocab_size)])
print("Matriks One-Hot Encoding untuk Seluruh Vocabulary:")
display(df_one_hot.astype(int))

Matriks One-Hot Encoding untuk Seluruh Vocabulary:


,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,dim_11,dim_12,dim_13,dim_14,dim_15,dim_16
algoritma,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
belajar,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
bola,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
buatan,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
cerdas,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
gol,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
informatika,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
kecerdasan,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
kemenangan,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
komputer,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0


## 3. Pembentukan Pasangan Data Latih (Target & Context Pairs) dengan Sliding Window

Model **Skip-gram** bekerja dengan prinsip: **diberikan satu kata target ($w_t$), model bertugas memprediksi kata-kata konteks ($w_{t+j}$) yang berada di sekitarnya dalam rentang jendela tertentu (*window size* $C$).**

Dengan **`WINDOW_SIZE = 3`**, untuk kata pada posisi $t$:
- Kata target: $w_t$
- Kata konteks: kata-kata yang berjarak maksimal 3 kata ke kiri dan 3 kata ke kanan:
  $$\{w_{t-3}, w_{t-2}, w_{t-1}\} \quad \mathbf{[w_t]} \quad \{w_{t+1}, w_{t+2}, w_{t+3}\}$$

> **Catatan Penting:** 
> Pasangan data latih disusun **satu baris untuk setiap pasangan tunggal (Target, Context)**. Jika suatu kata target memiliki 4 kata konteks di sekitarnya, maka kata target tersebut akan muncul sebanyak 4 kali di tabel.

In [4]:
WINDOW_SIZE = 3
training_pairs = []

for s_idx, sentence in enumerate(tokenized_sentences):
    for target_pos, target_word in enumerate(sentence):
        # Tentukan batas jendela konteks di sekitar target_pos
        start = max(0, target_pos - WINDOW_SIZE)
        end = min(len(sentence), target_pos + WINDOW_SIZE + 1)
        
        for context_pos in range(start, end):
            if context_pos != target_pos:
                context_word = sentence[context_pos]
                training_pairs.append({
                    'Kalimat_Ke': s_idx + 1,
                    'Target_Word': target_word,
                    'Target_Idx': word2idx[target_word],
                    'Context_Word': context_word,
                    'Context_Idx': word2idx[context_word]
                })

df_pairs = pd.DataFrame(training_pairs)
print(f"Total pasangan data latih (Target, Context) yang terbentuk: {len(df_pairs)} pasangan\n")

# Tampilkan 12 baris pertama agar terlihat transisi antar-kata target
print("Contoh 12 Pasangan Data Latih Pertama (df_pairs.head(12)):")
display(df_pairs.head(12))

Total pasangan data latih (Target, Context) yang terbentuk: 66 pasangan

Contoh 12 Pasangan Data Latih Pertama (df_pairs.head(12)):


,Kalimat_Ke,Target_Word,Target_Idx,Context_Word,Context_Idx
0,1,mahasiswa,10,informatika,6
1,1,mahasiswa,10,belajar,1
2,1,mahasiswa,10,kecerdasan,7
3,1,informatika,6,mahasiswa,10
4,1,informatika,6,belajar,1
5,1,informatika,6,kecerdasan,7
6,1,informatika,6,buatan,3
7,1,belajar,1,mahasiswa,10
8,1,belajar,1,informatika,6
9,1,belajar,1,kecerdasan,7


### 🔍 Pemeriksaan Khusus Kata Target (Contoh: Kata 'belajar')

Jika kita memfilter tabel data latih khusus untuk kata target **`belajar`** pada Kalimat 1 (*"mahasiswa informatika **belajar** kecerdasan buatan"*):
- Jarak -2: `mahasiswa`
- Jarak -1: `informatika`
- Jarak +1: `kecerdasan`
- Jarak +2: `buatan`

Keempat kata tersebut semuanya muncul sebagai konteks dari kata target **`belajar`**:

In [8]:
# Filter tabel khusus target kata 'belajar'
df_belajar = df_pairs[df_pairs['Target_Word'] == 'belajar']
print("Pasangan (Target, Context) Khusus Kata 'belajar':")
display(df_belajar)

# Tabel Ringkasan: Seluruh Kata Target dan Daftar Konteks Lengkapnya
df_summary_contexts = df_pairs.groupby('Target_Word')['Context_Word'].apply(list).reset_index()
df_summary_contexts.columns = ['Kata Target', f'Daftar Kata Konteks (Window = {WINDOW_SIZE})']
print("\nRingkasan Seluruh Kata Target Beserta Daftar Konteksnya:")
display(df_summary_contexts)

Pasangan (Target, Context) Khusus Kata 'belajar':


,Kalimat_Ke,Target_Word,Target_Idx,Context_Word,Context_Idx
7,1,belajar,1,mahasiswa,10
8,1,belajar,1,informatika,6
9,1,belajar,1,kecerdasan,7
10,1,belajar,1,buatan,3



Ringkasan Seluruh Kata Target Beserta Daftar Konteksnya:


,Kata Target,Daftar Kata Konteks (Window = 3)
0,algoritma,"[program, komputer, menggunakan, cerdas]"
1,belajar,"[mahasiswa, informatika, kecerdasan, buatan]"
2,bola,"[pemain, sepak, mencetak, gol, kemenangan]"
3,buatan,"[informatika, belajar, kecerdasan]"
4,cerdas,"[komputer, menggunakan, algoritma]"
5,gol,"[sepak, bola, mencetak, kemenangan, tim]"
6,informatika,"[mahasiswa, belajar, kecerdasan, buatan]"
7,kecerdasan,"[mahasiswa, informatika, belajar, buatan]"
8,kemenangan,"[bola, mencetak, gol, tim]"
9,komputer,"[program, menggunakan, algoritma, cerdas]"
